In [2]:
import random
import numpy as np
from igraph import Graph
from data_generation import add_initial_opinions, generate_base_network_LFR, generate_base_network_SBM
from measure_polarization import measure_network_polarization
from simulate_evolution import simulate_network_evolution
from utils import get_cross_density
from visualization import plot_ideological_polarization_LFR, plot_relational_polarization_LFR, plot_ideological_polarization_SBM, plot_relational_polarization_SBM
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx

In [ ]:
# GLOBAL PARAMS
iterations = 20
global_seed = 42

N = 500

issues = 5
p_agree = 0.2
seeds_per_community_ratio = 0.1

# LFR params
gamma_values = [2.5]
beta = 1.5
min_degree = 5
max_degree = min(50, N//5)
min_community = int(max_degree * 1.2)
max_community = N // 4

mu_values = np.linspace(0.1, 0.6, 11).tolist()
results = []
for gamma in gamma_values:
    for i, mu in enumerate(mu_values):
        iteration_results = [] 
        for it in range(iterations):
            print(f"Config: γ={gamma}, μ={mu:.2f} | Iteration {it+1}/{iterations}")

            curr_seed = global_seed + (int(gamma * 100) * 1000) + (i * iterations) + it
            g = None
            attempts = 0
    
            while g is None:
                try:
                    g, communities = generate_base_network_LFR(N, gamma=gamma, beta=beta, mu=mu, 
                                                       min_degree=min_degree, max_degree=max_degree, 
                                                       min_community=min_community, max_community=max_community, 
                                                       seed=curr_seed)
                except nx.ExceededMaxIterations:
                    attempts += 1
            
                    curr_seed += 9999  
                   
            
                    if attempts > 10:
                        raise RuntimeError(f"LFR impossible to converge after 10 distinct seed shifts.")
            g = add_initial_opinions(g, N, issues, communities, p_agree, seeds_per_community_ratio, seed=curr_seed)
            #if (i == 3 and it == 1):
                #visualize_social_layer(g, filename=f"graph_visuals/Initial Network Social (mu={mu:.2f}).jpg")
                #visualize_ideological_layer(g, filename=f"graph_visuals/Initial Network Ideological (mu={mu:.2f}).jpg")
            g = simulate_network_evolution(g, N, issues, steps=20, p_rewire=0.05)
            #if (i == 3 and it == 1):
                #visualize_social_layer(g, filename=f"graph_visuals/Evolved Network Social (mu={mu:.2f}).jpg")
                #visualize_ideological_layer(g, filename=f"graph_visuals/Evolved Network Ideological (mu={mu:.2f}).jpg")
            metrics = measure_network_polarization(g)
            density = get_cross_density(g)

            # iteration_results.append({
            #     'mu': mu,
            #     'relational_polarization': metrics['relational_polarization'],
            #     'ideological_polarization': metrics['ideological_polarization']
            # })
        
        
    #df_step = pd.DataFrame(iteration_results)
            results.append({
                'mu': mu,
                'relational_polarization':  metrics['relational_polarization'],
                'ideological_polarization': metrics['ideological_polarization'],
                'communities': communities,
                'actors': N,
                'gamma': gamma,
                'issues': issues,
            })
df_final_summary = pd.DataFrame(results)

df_grouped = df_final_summary.groupby(['gamma', 'mu']).agg(
        relational_polarization_mean=('relational_polarization', 'mean'),
        relational_polarization_std=('relational_polarization', 'std'),
        ideological_polarization_mean=('ideological_polarization', 'mean'),
        ideological_polarization_std=('ideological_polarization', 'std'),
        avg_communities=('communities', 'mean')
    ).reset_index()
   
df_grouped['actors'] = N
df_grouped['communities'] = communities
    
aggregated_csv_path = f"results_csv/lfr_simulations_aggregated_n{N}_c{communities}.csv"
df_csv = df_grouped.to_csv(aggregated_csv_path, index=False)    
print(f"Results saved to {aggregated_csv_path}")
#plot_relational_polarization(df_grouped)
#plot_ideological_polarization(df_grouped)

In [ ]:

aggregated_csv_path = "results_csv/lfr_simulations_aggregated_n1000_c9.csv"
df_plot = pd.read_csv(aggregated_csv_path)
plot_relational_polarization_LFR(df_plot)
plot_ideological_polarization_LFR(df_plot)

Ideological polarization plot saved to results/ideological_polarization_n1000_c9.png


In [ ]:
#SBM Parameters
N = 1000
iterations = 10
communities = [2, 4, 8]

deg_in = 20
ratio_values = np.linspace(0.1, 0.6, 11).tolist()

for comm_no in communities:
    p_in = deg_in / N * comm_no

    for i, ratio in enumerate(ratio_values):
        deg_out = ratio * deg_in
        p_out = deg_out / N * comm_no

        iteration_results = [] 
        for it in range(iterations):
            print(f"Config: comm={comm_no}, ratio={ratio:.2f} | Iteration {it+1}/{iterations}")

            curr_seed = global_seed + (int(comm_no * 100) * 1000) + (i * iterations) + it
            g = None
            attempts = 0
            g = generate_base_network_SBM(N, comm_no, p_in, p_out, seed=curr_seed)
           
            g = add_initial_opinions(g, N, issues, comm_no, p_agree, seeds_per_community_ratio, seed=curr_seed)
            #if (i == 3 and it == 1):
                #visualize_social_layer(g, filename=f"graph_visuals/Initial Network Social (mu={mu:.2f}).jpg")
                #visualize_ideological_layer(g, filename=f"graph_visuals/Initial Network Ideological (mu={mu:.2f}).jpg")
            g = simulate_network_evolution(g, N, issues, steps=10, p_rewire=0.05)
            #if (i == 3 and it == 1):
                #visualize_social_layer(g, filename=f"graph_visuals/Evolved Network Social (mu={mu:.2f}).jpg")
                #visualize_ideological_layer(g, filename=f"graph_visuals/Evolved Network Ideological (mu={mu:.2f}).jpg")
            metrics = measure_network_polarization(g)
            density = get_cross_density(g)

            # iteration_results.append({
            #     'mu': mu,
            #     'relational_polarization': metrics['relational_polarization'],
            #     'ideological_polarization': metrics['ideological_polarization']
            # })
        
        
    #df_step = pd.DataFrame(iteration_results)
            results.append({
                'ratio': ratio,
                'relational_polarization':  metrics['relational_polarization'],
                'ideological_polarization': metrics['ideological_polarization'],
                'communities': comm_no,
                'actors': N,
                'issues': issues,
            })
df_final_summary = pd.DataFrame(results)

df_grouped = df_final_summary.groupby(['communities', 'ratio']).agg(
        relational_polarization_mean=('relational_polarization', 'mean'),
        relational_polarization_std=('relational_polarization', 'std'),
        ideological_polarization_mean=('ideological_polarization', 'mean'),
        ideological_polarization_std=('ideological_polarization', 'std'),
        avg_communities=('communities', 'mean')
    ).reset_index()
   
df_grouped['actors'] = N
    
aggregated_csv_path = f"results_csv/sbm_simulations_aggregated_n{N}.csv"
df_csv = df_grouped.to_csv(aggregated_csv_path, index=False)    
print(f"Results saved to {aggregated_csv_path}")

In [ ]:
aggregated_csv_path = "results_csv/sbm_simulations_aggregated_n1000.csv"
df_plot = pd.read_csv(aggregated_csv_path)
plot_relational_polarization_SBM(df_plot)
plot_ideological_polarization_SBM(df_plot)